# LAB6 — RAGAS: Comparação Faithfulness entre CRAG e Advanced RAG

---

| Campo | Detalhe |
|---|---|
| **Aula** | Aula 8 — Avaliação e Observabilidade em RAG |
| **Lab** | LAB6 — RAGAS: CRAG vs Advanced RAG (Baseline) |
| **Objetivo** | Comparar a métrica Faithfulness entre CRAG e Advanced RAG usando o framework RAGAS |
| **Duração estimada** | 60–90 minutos |
| **Pré-requisitos** | LAB2, LAB3 e LAB5 concluídos; chave OpenAI API válida |

---

## Contexto Teórico

**RAGAS** (Retrieval Augmented Generation Assessment) é um framework de avaliação automática para sistemas RAG. Ele mensura a qualidade usando LLMs como avaliadores, sem necessidade de julgamento humano em larga escala.

### As 4 Métricas Principais do RAGAS:

| Métrica | Pergunta respondida | Como é calculada |
|---|---|---|
| **Faithfulness** | A resposta é fiel aos contextos recuperados? | Proporção de afirmações da resposta com suporte nos contextos |
| **Answer Relevancy** | A resposta é relevante para a pergunta? | Similaridade semântica entre resposta e pergunta regenerada |
| **Context Recall** | Os contextos cobrem a ground truth? | Proporção da ground truth coberta pelos contextos |
| **Context Precision** | Os contextos recuperados são precisos? | Proporção de contextos que contribuem para a resposta correta |

### Foco deste Lab: **Faithfulness**

**Faithfulness** mede se a resposta gerada pelo LLM é "fiel" às fontes recuperadas. Uma resposta com alta faithfulness:
- Não contém alucinações (informações inventadas pelo LLM)
- Todas as afirmações podem ser verificadas nos documentos de contexto

**Fórmula:**
```
Faithfulness = (afirmações com suporte nos contextos) / (total de afirmações na resposta)
```

**Hipótese deste lab:** O CRAG deve ter maior Faithfulness que o Advanced RAG baseline, pois o CRAG avalia e filtra documentos irrelevantes antes de passar o contexto ao LLM — reduzindo o risco de alucinações.

## 1. Instalação e Configuração

In [ ]:
# Instalar todas as dependências necessárias
# ragas: framework de avaliação para sistemas RAG
# chromadb: vector store local para o baseline Advanced RAG
# sentence-transformers: modelo de embeddings para ChromaDB
# datasets: formato HuggingFace para datasets RAGAS
%pip install ragas langchain langchain-openai chromadb sentence-transformers datasets -q

In [ ]:
import os
from google.colab import userdata

# Configurar chave OpenAI (necessária para RAGAS usar LLM como avaliador)
try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✓ OPENAI_API_KEY carregada")
except Exception:
    os.environ["OPENAI_API_KEY"] = "sk-..."  # Substitua pela sua chave
    print("⚠ OPENAI_API_KEY definida manualmente")

# Suprimir warnings desnecessários para melhor leitura do notebook
import warnings
warnings.filterwarnings("ignore")

print("\n✓ Configuração concluída")

## 3. Definir as 5 Perguntas de Avaliação

Usamos o mesmo conjunto de perguntas do arquivo `datasets/queries_teste_aula8.json` para garantir comparabilidade entre sistemas. Cada pergunta tem uma **ground truth** — a resposta correta esperada — usada pelo RAGAS como referência.

In [ ]:
# Definir as 5 perguntas de avaliação com ground truth
# Estas perguntas cobrem os principais temas da base de conhecimento
PERGUNTAS_AVALIACAO = [
    {
        "id": 1,
        "question": "Qual é o prazo prescricional para ação de indenização por danos morais?",
        "ground_truth": "O prazo prescricional para ação de indenização por danos morais é de 3 (três) anos, "
                       "conforme art. 206, §3º, V do Código Civil brasileiro."
    },
    {
        "id": 2,
        "question": "O que é ato ilícito segundo o Código Civil brasileiro?",
        "ground_truth": "Ato ilícito é a ação ou omissão voluntária, negligência ou imprudência que viola "
                       "direito e causa dano a outrem, mesmo que exclusivamente moral, conforme art. 186 do "
                       "Código Civil. O abuso de direito também constitui ato ilícito (art. 187)."
    },
    {
        "id": 3,
        "question": "O que caracteriza o flagrante delito no direito processual penal brasileiro?",
        "ground_truth": "Flagrante delito é a situação em que o agente está cometendo, acaba de cometer ou "
                       "é perseguido logo após cometer infração penal, conforme art. 302 do CPP. "
                       "Existem 4 modalidades: próprio, impróprio, presumido e ficto."
    },
    {
        "id": 4,
        "question": "A pessoa jurídica pode ser vítima de dano moral?",
        "ground_truth": "Sim. O STJ consolidou na Súmula 227 que a pessoa jurídica pode sofrer dano moral, "
                       "especialmente quanto à sua honra objetiva — reputação e credibilidade no mercado."
    },
    {
        "id": 5,
        "question": "O que configura improbidade administrativa por enriquecimento ilícito?",
        "ground_truth": "Constitui improbidade por enriquecimento ilícito auferir qualquer vantagem patrimonial "
                       "indevida em razão do exercício de cargo ou função pública, conforme art. 9º da "
                       "Lei 8.429/1992 (com alterações da Lei 14.230/2021). Exige-se dolo específico."
    }
]

print(f"✓ {len(PERGUNTAS_AVALIACAO)} perguntas de avaliação definidas:")
for p in PERGUNTAS_AVALIACAO:
    print(f"  {p['id']}. {p['question'][:70]}")

## 4. Base de Conhecimento e Vector Store

Criamos a base de conhecimento jurídica e indexamos no ChromaDB para ambos os sistemas.

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

# ===== CORPUS JURÍDICO PARA INDEXAÇÃO =====
# Documentos da base de conhecimento — cobrindo os temas das 5 perguntas
CORPUS_JURIDICO = [
    # Danos morais e prescrição
    Document(
        page_content="""Art. 206, §3º, V do Código Civil: Prescreve em 3 (três) anos a pretensão 
        de reparação civil. O STJ consolida que o prazo prescricional da ação de indenização por 
        danos morais conta-se da data em que o ofendido toma ciência do fato lesivo.""",
        metadata={"source": "Codigo Civil", "artigo": "206", "tema": "prescricao"}
    ),
    # Ato ilícito
    Document(
        page_content="""Art. 186 do Código Civil: 'Aquele que, por ação ou omissão voluntária, negligência 
        ou imprudência, violar direito e causar dano a outrem, ainda que exclusivamente moral, comete 
        ato ilícito.' Art. 187: Também comete ato ilícito o titular de um direito que, ao exercê-lo, 
        excede manifestamente os limites impostos pelo seu fim econômico ou social, pela boa-fé 
        ou pelos bons costumes (abuso de direito).""",
        metadata={"source": "Codigo Civil", "artigo": "186-187", "tema": "ato_ilicito"}
    ),
    # Flagrante delito
    Document(
        page_content="""Art. 302 do CPP — Hipóteses de flagrante: I - está cometendo a infração penal 
        (flagrante próprio); II - acaba de cometê-la (flagrante impróprio ou quase-flagrante); 
        III - é perseguido logo após pela autoridade, pelo ofendido ou por qualquer pessoa 
        (flagrante presumido); IV - é encontrado logo depois com instrumentos ou objetos que façam 
        presumir ser ele o autor da infração.""",
        metadata={"source": "CPP", "artigo": "302", "tema": "flagrante"}
    ),
    # Pessoa jurídica e dano moral
    Document(
        page_content="""Súmula 227 do STJ: 'A pessoa jurídica pode sofrer dano moral.' O fundamento 
        reside na proteção à honra objetiva da empresa — sua reputação, imagem e credibilidade 
        perante o mercado. Não se confunde com o dano moral subjetivo (sofrimento psíquico), 
        próprio das pessoas físicas.""",
        metadata={"source": "STJ", "sumula": "227", "tema": "dano_moral_pj"}
    ),
    # Improbidade administrativa
    Document(
        page_content="""Lei 8.429/1992, art. 9º (com redação da Lei 14.230/2021): Constitui ato de 
        improbidade administrativa importando enriquecimento ilícito auferir qualquer tipo de 
        vantagem patrimonial indevida em razão do exercício de cargo, mandato, função, emprego 
        ou atividade nas entidades referidas no art. 1º desta lei. A Lei 14.230/2021 exige 
        comprovação de dolo específico para caracterização do ato.""",
        metadata={"source": "Lei 8429/1992", "artigo": "9", "tema": "improbidade"}
    ),
    # Documentos de ruído (para testar robustez do CRAG)
    Document(
        page_content="""O processo civil brasileiro é regulado pelo Código de Processo Civil (Lei 13.105/2015). 
        O CPC estabelece as regras de competência, procedimentos e recursos para ações cíveis. 
        A tutela provisória permite medidas urgentes antes do julgamento final.""",
        metadata={"source": "CPC", "tema": "processo_civil"}
    ),
    Document(
        page_content="""O direito tributário brasileiro é regido pelo Código Tributário Nacional (CTN - Lei 5.172/1966). 
        O CTN define os princípios da legalidade, anterioridade e irretroatividade tributária. 
        A obrigação tributária surge com a ocorrência do fato gerador.""",
        metadata={"source": "CTN", "tema": "tributario"}
    ),
]

print(f"✓ Corpus jurídico: {len(CORPUS_JURIDICO)} documentos")
print(f"  Documentos temáticos: 5 | Documentos de ruído: 2")

# ===== INDEXAR NO CHROMADB =====
# Usar embeddings OpenAI para representação semântica dos documentos
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("\nCriando vector store ChromaDB...")
vectorstore = Chroma.from_documents(
    documents=CORPUS_JURIDICO,
    embedding=embeddings,
    collection_name="juridico_aula8"  # Nome da coleção no ChromaDB
)
print(f"✓ ChromaDB criado com {len(CORPUS_JURIDICO)} documentos indexados")

## 4. Advanced RAG Baseline — Retrieval Simples sem Avaliação de Qualidade

O **Advanced RAG baseline** recupera os top-k documentos mais similares e os passa diretamente ao LLM, **sem avaliar a relevância**. É o padrão mais comum em implementações RAG simples.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# LLM para geração de respostas
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Retriever simples: busca os k=3 documentos mais similares por cosseno
# SEM avaliação de relevância — todos os documentos recuperados são usados
retriever_simples = vectorstore.as_retriever(
    search_type="similarity",  # Similaridade de cosseno
    search_kwargs={"k": 3}     # Retornar os 3 mais similares
)

# Prompt para geração de resposta
prompt_rag = ChatPromptTemplate.from_messages([
    ("system", """Você é um especialista em direito brasileiro.
Responda à pergunta usando APENAS as informações do contexto fornecido.
Se o contexto não contiver a informação, diga 'Não encontrei essa informação na base de dados.'"""),
    ("human", "Contexto:\n{context}\n\nPergunta: {question}")
])

def advanced_rag_baseline(question: str) -> dict:
    """
    Advanced RAG baseline: retrieval simples (k=3) + geração direta.
    NÃO avalia a qualidade dos documentos recuperados.
    
    Args:
        question: Pergunta jurídica do usuário
    
    Returns:
        dict com answer e contexts
    """
    # Recuperar os 3 documentos mais similares
    docs = retriever_simples.invoke(question)
    
    # Concatenar documentos como contexto (sem filtro de relevância)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # Gerar resposta com o LLM
    chain = prompt_rag | llm
    resposta = chain.invoke({"context": context, "question": question})
    
    return {
        "answer": resposta.content,
        "contexts": [doc.page_content for doc in docs]
    }


# Testar o baseline com a primeira pergunta
print("Testando Advanced RAG baseline...")
teste_baseline = advanced_rag_baseline(PERGUNTAS_AVALIACAO[0]["question"])
print(f"Pergunta: {PERGUNTAS_AVALIACAO[0]['question']}")
print(f"Resposta: {teste_baseline['answer'][:200]}...")
print(f"Docs recuperados: {len(teste_baseline['contexts'])}")
print("\n✓ Baseline funcionando")

## 5. CRAG — Com Avaliador de Relevância e Fallback

O CRAG adiciona uma camada de avaliação: antes de usar os documentos, verificamos se são relevantes. Documentos irrelevantes são descartados, reduzindo ruído no contexto.

In [ ]:
from pydantic import BaseModel, Field

# Schema de avaliação de relevância (igual ao LAB3)
class RelevanceGrade(BaseModel):
    """Avaliação de relevância de um documento para uma pergunta."""
    relevante: str = Field(description="'sim' se o documento é relevante, 'não' se irrelevante")
    justificativa: str = Field(description="Breve explicação da decisão")

# Avaliador de relevância com saída estruturada
llm_avaliador = ChatOpenAI(model="gpt-4o-mini", temperature=0)
avaliador_estruturado = llm_avaliador.with_structured_output(RelevanceGrade)

# Prompt do avaliador
prompt_avaliador = ChatPromptTemplate.from_messages([
    ("system", """Você é um avaliador de relevância jurídica. 
Determine se o documento é relevante para responder a pergunta.
Responda com 'sim' para relevante ou 'não' para irrelevante."""),
    ("human", "Pergunta: {question}\n\nDocumento: {document}")
])
chain_avaliador = prompt_avaliador | avaliador_estruturado


def crag_sistema(question: str) -> dict:
    """
    CRAG simplificado: retrieval + avaliação de relevância + fallback.
    Usa apenas documentos avaliados como relevantes para gerar a resposta.
    
    Args:
        question: Pergunta jurídica do usuário
    
    Returns:
        dict com answer, contexts e metadados de avaliação
    """
    # ---- ETAPA 1: Recuperar documentos (mesmo retriever do baseline) ----
    docs = retriever_simples.invoke(question)
    
    # ---- ETAPA 2: Avaliar relevância de cada documento ----
    docs_relevantes = []
    scores_relevancia = []
    
    for doc in docs:
        # Invocar avaliador para este par (pergunta, documento)
        avaliacao = chain_avaliador.invoke({
            "question": question,
            "document": doc.page_content
        })
        
        # Verificar se o documento é relevante
        eh_relevante = avaliacao.relevante.lower() in ["sim", "yes", "s"]
        scores_relevancia.append(1 if eh_relevante else 0)
        
        if eh_relevante:
            docs_relevantes.append(doc)
    
    # ---- ETAPA 3: Fallback se nenhum doc for relevante ----
    # Em produção, ativaria web search (Tavily do LAB4)
    # Aqui usamos todos os docs como fallback para não ficar sem contexto
    if not docs_relevantes:
        docs_para_gerar = docs  # Fallback: usar todos
        rota = "fallback"
    else:
        docs_para_gerar = docs_relevantes  # Usar apenas os relevantes
        rota = "local_filtrado"
    
    # ---- ETAPA 4: Gerar resposta com contexto filtrado ----
    context = "\n\n".join([doc.page_content for doc in docs_para_gerar])
    chain_gen = prompt_rag | llm
    resposta = chain_gen.invoke({"context": context, "question": question})
    
    return {
        "answer": resposta.content,
        "contexts": [doc.page_content for doc in docs_para_gerar],
        "rota": rota,
        "docs_originais": len(docs),
        "docs_relevantes": len(docs_relevantes),
        "scores_relevancia": scores_relevancia
    }


# Testar o CRAG com a primeira pergunta
print("Testando CRAG...")
teste_crag = crag_sistema(PERGUNTAS_AVALIACAO[0]["question"])
print(f"Pergunta: {PERGUNTAS_AVALIACAO[0]['question']}")
print(f"Docs recuperados: {teste_crag['docs_originais']} | Docs relevantes: {teste_crag['docs_relevantes']}")
print(f"Rota: {teste_crag['rota']}")
print(f"Resposta: {teste_crag['answer'][:200]}...")
print("\n✓ CRAG funcionando")

## 6. Coletar Resultados para Ambos os Sistemas

Rodamos as 5 perguntas em ambos os sistemas e coletamos (question, answer, contexts, ground_truth) para avaliação RAGAS.

In [ ]:
import time

# Listas para armazenar os dados de cada sistema
dados_baseline = {"question": [], "answer": [], "contexts": [], "ground_truth": []}
dados_crag = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

# Metadados adicionais do CRAG (para análise qualitativa)
meta_crag = []

print("Coletando resultados para avaliação RAGAS...")
print("=" * 70)

for pergunta_info in PERGUNTAS_AVALIACAO:
    question = pergunta_info["question"]
    ground_truth = pergunta_info["ground_truth"]
    
    print(f"\nPergunta {pergunta_info['id']}: {question[:60]}..." if len(question) > 60 else f"\nPergunta {pergunta_info['id']}: {question}")
    
    # ---- ADVANCED RAG BASELINE ----
    print("  [Baseline] Executando...")
    resultado_baseline = advanced_rag_baseline(question)
    
    # Adicionar aos dados do baseline
    dados_baseline["question"].append(question)
    dados_baseline["answer"].append(resultado_baseline["answer"])
    dados_baseline["contexts"].append(resultado_baseline["contexts"])
    dados_baseline["ground_truth"].append(ground_truth)
    
    time.sleep(0.5)  # Respeitar rate limits
    
    # ---- CRAG ----
    print("  [CRAG] Executando...")
    resultado_crag = crag_sistema(question)
    
    # Adicionar aos dados do CRAG
    dados_crag["question"].append(question)
    dados_crag["answer"].append(resultado_crag["answer"])
    dados_crag["contexts"].append(resultado_crag["contexts"])
    dados_crag["ground_truth"].append(ground_truth)
    
    # Salvar metadados do CRAG para análise qualitativa
    meta_crag.append({
        "pergunta_id": pergunta_info["id"],
        "rota": resultado_crag["rota"],
        "docs_originais": resultado_crag["docs_originais"],
        "docs_relevantes": resultado_crag["docs_relevantes"],
        "reducao_contexto": resultado_crag["docs_originais"] - resultado_crag["docs_relevantes"]
    })
    
    print(f"  Baseline: {len(resultado_baseline['contexts'])} docs | CRAG: {resultado_crag['docs_relevantes']}/{resultado_crag['docs_originais']} docs relevantes")
    
    time.sleep(1)  # Pausa entre perguntas

print("\n" + "=" * 70)
print(f"✓ Coleta concluída: {len(dados_baseline['question'])} perguntas em cada sistema")

## 7. Criar Datasets RAGAS

O RAGAS espera um formato específico: `Dataset.from_dict()` com campos `question`, `answer`, `contexts` e `ground_truth`.

In [ ]:
from datasets import Dataset

# Criar dataset HuggingFace para o baseline
dataset_baseline = Dataset.from_dict(dados_baseline)

# Criar dataset HuggingFace para o CRAG
dataset_crag = Dataset.from_dict(dados_crag)

print("Datasets RAGAS criados com sucesso:")
print(f"  Dataset Baseline: {len(dataset_baseline)} exemplos")
print(f"  Dataset CRAG:     {len(dataset_crag)} exemplos")

# Verificar a estrutura do primeiro exemplo
print("\nEstrutura do dataset (primeiro exemplo do CRAG):")
print(f"  question:     {dataset_crag[0]['question'][:60]}...")
print(f"  answer:       {dataset_crag[0]['answer'][:80]}...")
print(f"  contexts:     {len(dataset_crag[0]['contexts'])} documentos")
print(f"  ground_truth: {dataset_crag[0]['ground_truth'][:60]}...")

## 8. Rodar Avaliação RAGAS para Ambos os Sistemas

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Configurar o LLM e embeddings que o RAGAS usará como avaliadores
# RAGAS usa LLMs internamente para calcular as métricas
llm_ragas = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings_ragas = OpenAIEmbeddings(model="text-embedding-3-small")

# Métricas a calcular
# - faithfulness: proporção de afirmações suportadas pelos contextos
# - answer_relevancy: relevância da resposta para a pergunta
METRICAS = [faithfulness, answer_relevancy]

print("Executando avaliação RAGAS...")
print("(Pode demorar 2-5 minutos devido às chamadas LLM internas)")
print()

# ---- AVALIAR BASELINE ----
print("[1/2] Avaliando Advanced RAG Baseline...")
resultado_ragas_baseline = evaluate(
    dataset=dataset_baseline,
    metrics=METRICAS,
    llm=llm_ragas,
    embeddings=embeddings_ragas
)
print(f"  Baseline concluído ✓")
print(f"  Faithfulness:     {resultado_ragas_baseline['faithfulness']:.4f}")
print(f"  Answer Relevancy: {resultado_ragas_baseline['answer_relevancy']:.4f}")

# ---- AVALIAR CRAG ----
print("\n[2/2] Avaliando CRAG...")
resultado_ragas_crag = evaluate(
    dataset=dataset_crag,
    metrics=METRICAS,
    llm=llm_ragas,
    embeddings=embeddings_ragas
)
print(f"  CRAG concluído ✓")
print(f"  Faithfulness:     {resultado_ragas_crag['faithfulness']:.4f}")
print(f"  Answer Relevancy: {resultado_ragas_crag['answer_relevancy']:.4f}")

print("\n✓ Avaliação RAGAS concluída para ambos os sistemas")

## 9. DataFrame Comparativo com Delta e Melhora %

In [ ]:
import pandas as pd

# Extrair scores escalares das métricas
scores_baseline = {
    "faithfulness": resultado_ragas_baseline["faithfulness"],
    "answer_relevancy": resultado_ragas_baseline["answer_relevancy"]
}

scores_crag = {
    "faithfulness": resultado_ragas_crag["faithfulness"],
    "answer_relevancy": resultado_ragas_crag["answer_relevancy"]
}

# Construir tabela comparativa
linhas = []
for metrica in ["faithfulness", "answer_relevancy"]:
    val_baseline = scores_baseline[metrica]
    val_crag = scores_crag[metrica]
    delta = val_crag - val_baseline
    melhora_pct = (delta / val_baseline * 100) if val_baseline > 0 else 0
    
    linhas.append({
        "Métrica": metrica.replace("_", " ").title(),
        "Advanced RAG": round(val_baseline, 4),
        "CRAG": round(val_crag, 4),
        "Delta": round(delta, 4),
        "Melhora %": f"{melhora_pct:+.1f}%",
        "CRAG Melhor?": "Sim ✓" if delta > 0 else ("Empate" if delta == 0 else "Não ✗")
    })

df_comparativo = pd.DataFrame(linhas)

print("=" * 80)
print("COMPARATIVO RAGAS: ADVANCED RAG vs CRAG")
print("=" * 80)
print(df_comparativo.to_string(index=False))
print("=" * 80)

# Análise textual automática
faith_base = scores_baseline["faithfulness"]
faith_crag = scores_crag["faithfulness"]
delta_faith = faith_crag - faith_base

print("\nANÁLISE:")
if delta_faith > 0.05:
    print(f"  ✓ CRAG apresenta Faithfulness {delta_faith:.3f} MAIOR que o baseline")
    print(f"    Isso indica que o CRAG gera menos alucinações graças ao filtro de relevância")
elif delta_faith < -0.05:
    print(f"  ✗ CRAG apresenta Faithfulness {abs(delta_faith):.3f} MENOR que o baseline")
    print(f"    Considere revisar o threshold do avaliador de relevância")
else:
    print(f"  ≈ Diferença de Faithfulness é pequena ({delta_faith:.3f})")
    print(f"    Os sistemas têm desempenho similar neste conjunto de 5 perguntas")

## 10. Gráfico de Barras Comparativo

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Dados para o gráfico
metricas_nomes = ["Faithfulness", "Answer\nRelevancy"]
vals_baseline = [scores_baseline["faithfulness"], scores_baseline["answer_relevancy"]]
vals_crag = [scores_crag["faithfulness"], scores_crag["answer_relevancy"]]

x = np.arange(len(metricas_nomes))
largura = 0.35  # Largura das barras

fig, ax = plt.subplots(figsize=(10, 6))

# Criar barras lado a lado
barras_baseline = ax.bar(
    x - largura/2, vals_baseline, largura,
    label="Advanced RAG (Baseline)",
    color="steelblue", edgecolor="black", alpha=0.85
)
barras_crag = ax.bar(
    x + largura/2, vals_crag, largura,
    label="CRAG",
    color="darkorange", edgecolor="black", alpha=0.85
)

# Adicionar valores nas barras
def adicionar_labels(bars):
    for bar in bars:
        height = bar.get_height()
        ax.annotate(
            f"{height:.3f}",
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 5),
            textcoords="offset points",
            ha="center", va="bottom",
            fontsize=11, fontweight="bold"
        )

adicionar_labels(barras_baseline)
adicionar_labels(barras_crag)

# Adicionar setas indicando melhora/piora
for i, (v_base, v_crag) in enumerate(zip(vals_baseline, vals_crag)):
    delta = v_crag - v_base
    cor_delta = "green" if delta >= 0 else "red"
    seta = "↑" if delta > 0 else ("↓" if delta < 0 else "→")
    ax.text(i, max(v_base, v_crag) + 0.08, f"{seta}{abs(delta):.3f}",
            ha="center", fontsize=12, color=cor_delta, fontweight="bold")

# Formatação do gráfico
ax.set_xlabel("Métricas RAGAS", fontsize=12)
ax.set_ylabel("Score (0-1)", fontsize=12)
ax.set_title(
    "Comparativo RAGAS: Advanced RAG Baseline vs CRAG\n"
    "(contexto jurídico brasileiro — 5 perguntas)",
    fontsize=13
)
ax.set_xticks(x)
ax.set_xticklabels(metricas_nomes, fontsize=12)
ax.set_ylim(0, 1.2)
ax.legend(fontsize=11, loc="upper right")
ax.grid(axis="y", alpha=0.3)
ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.5, label="Score máximo")

# Nota de rodapé com informações do experimento
fig.text(0.5, -0.02,
         f"Avaliação com gpt-4o-mini | {len(PERGUNTAS_AVALIACAO)} perguntas | ChromaDB k=3",
         ha="center", fontsize=9, color="gray")

plt.tight_layout()
plt.savefig("lab6_ragas_comparativo.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Gráfico comparativo salvo como lab6_ragas_comparativo.png")

## 11. Análise Qualitativa: Para Quais Perguntas o CRAG Melhorou Mais?

Identificamos as perguntas onde a filtragem de documentos do CRAG fez maior diferença.

In [ ]:
# Analisar quais perguntas tiveram maior redução de contexto no CRAG
df_meta = pd.DataFrame(meta_crag)

print("ANÁLISE QUALITATIVA — CRAG por Pergunta")
print("=" * 80)

# Merge com perguntas
df_meta["pergunta"] = [p["question"][:55] + "..." for p in PERGUNTAS_AVALIACAO]

# Calcular taxa de filtro (proporção de docs removidos)
df_meta["taxa_filtro"] = df_meta["reducao_contexto"] / df_meta["docs_originais"]

print(df_meta[["pergunta_id", "pergunta", "docs_originais", 
               "docs_relevantes", "reducao_contexto", "rota"]].to_string(index=False))

print("\n--- Análise de Filtro ---")
print(f"Média de docs removidos por query: {df_meta['reducao_contexto'].mean():.1f}")
print(f"Query com maior filtro: Pergunta {df_meta.loc[df_meta['reducao_contexto'].idxmax(), 'pergunta_id']}")
print(f"  ({df_meta['reducao_contexto'].max()} docs removidos)")

# Comparação resposta por resposta
print("\n" + "=" * 80)
print("COMPARAÇÃO DE RESPOSTAS POR PERGUNTA")
print("=" * 80)

for i, (p_info, resp_base, resp_crag) in enumerate(zip(
    PERGUNTAS_AVALIACAO, dados_baseline["answer"], dados_crag["answer"]
)):
    print(f"\n[Pergunta {p_info['id']}] {p_info['question'][:70]}")
    print(f"  Ground Truth: {p_info['ground_truth'][:100]}...")
    print(f"  Baseline: {resp_base[:120]}...")
    print(f"  CRAG:     {resp_crag[:120]}...")
    
    # Verificar se o CRAG reduziu o contexto
    meta = df_meta[df_meta["pergunta_id"] == p_info["id"]].iloc[0]
    if meta["reducao_contexto"] > 0:
        print(f"  → CRAG filtrou {meta['reducao_contexto']} doc(s) irrelevante(s) — contexto mais limpo")
    else:
        print(f"  → CRAG manteve todos os documentos (todos considerados relevantes)")

## 12. Célula de Conclusão — Template para Documentação

**Instruções:** Preencha o template abaixo com os resultados obtidos em sua execução do lab. Este template pode ser usado como parte do relatório de avaliação da Aula 8.

In [ ]:
# ============================================================
# TEMPLATE DE DOCUMENTAÇÃO — Preencha com seus resultados
# ============================================================

# Recuperar valores calculados anteriormente
faith_baseline = scores_baseline.get("faithfulness", 0)
faith_crag = scores_crag.get("faithfulness", 0)
ansrel_baseline = scores_baseline.get("answer_relevancy", 0)
ansrel_crag = scores_crag.get("answer_relevancy", 0)

template = f"""
========================================================
RELATÓRIO DE AVALIAÇÃO — LAB6
MBA em RAG & CAG Aplicados a Direito e Segurança Pública
Aula 8 — Avaliação e Observabilidade em RAG
========================================================

Data de execução: [PREENCHA A DATA]
Aluno: [PREENCHA SEU NOME]

CONFIGURAÇÃO DO EXPERIMENTO:
  Modelo LLM: gpt-4o-mini
  Embeddings: text-embedding-3-small
  Vector Store: ChromaDB (k=3)
  Perguntas avaliadas: {len(PERGUNTAS_AVALIACAO)}
  Documentos na base: {len(CORPUS_JURIDICO)}

RESULTADOS RAGAS:
  ┌─────────────────────┬──────────────┬──────────┬──────────┐
  │ Métrica             │ Advanced RAG │   CRAG   │  Delta   │
  ├─────────────────────┼──────────────┼──────────┼──────────┤
  │ Faithfulness        │   {faith_baseline:.4f}   │  {faith_crag:.4f}  │  {faith_crag-faith_baseline:+.4f}  │
  │ Answer Relevancy    │   {ansrel_baseline:.4f}   │  {ansrel_crag:.4f}  │  {ansrel_crag-ansrel_baseline:+.4f}  │
  └─────────────────────┴──────────────┴──────────┴──────────┘

ANÁLISE DO ALUNO:
  1. O CRAG apresentou maior Faithfulness? [SIM/NÃO]
     Justificativa: [PREENCHA]

  2. Para quais perguntas a diferença foi mais significativa?
     [PREENCHA COM AS PERGUNTAS E MOTIVOS]

  3. O filtro de relevância do CRAG funcionou conforme esperado?
     Média de docs filtrados por query: {df_meta['reducao_contexto'].mean():.1f}
     Comentário: [PREENCHA]

  4. Limitações identificadas neste experimento:
     - Tamanho do conjunto de avaliação (apenas 5 perguntas)
     - [ADICIONE OUTRAS LIMITAÇÕES]

  5. Próximos passos para melhorar o sistema:
     - [PREENCHA]

CONCLUSÃO:
  [ESCREVA UM PARÁGRAFO RESUMINDO SUA ANÁLISE COMPARATIVA]

========================================================
"""

print(template)

# Salvar relatório em arquivo texto
with open("lab6_relatorio_avaliacao.txt", "w", encoding="utf-8") as f:
    f.write(template)

print("✓ Template salvo em lab6_relatorio_avaliacao.txt")
print("  Preencha os campos marcados com [PREENCHA] com suas análises")

## Conclusão

### O que aprendemos neste lab:

1. **RAGAS** automatiza a avaliação de sistemas RAG usando LLMs como avaliadores — escalável para grandes volumes de testes
2. **Faithfulness** é a métrica mais crítica para aplicações jurídicas: alucinações podem causar prejuízo real
3. **CRAG reduz alucinações** ao filtrar documentos irrelevantes, produzindo contexto mais limpo para o LLM
4. A avaliação sistemática revela **quais tipos de perguntas** cada sistema responde melhor

### Perguntas de Reflexão:

1. Por que Faithfulness é especialmente crítica em sistemas RAG para **uso jurídico**? Que impactos uma resposta infiel poderia ter?

2. Se você aumentasse o corpus para 100 documentos (com muito ruído), a diferença entre CRAG e baseline seria maior ou menor? Por quê?

3. O RAGAS usa LLMs para calcular as métricas. Isso significa que os resultados podem variar entre execuções. Como você tornaria a avaliação mais **estável e reproduzível**?

4. Quais outras métricas do RAGAS seriam mais relevantes para o contexto de **segurança pública**? (Context Recall? Context Precision?)

5. Como você integraria a avaliação RAGAS em um **pipeline de CI/CD** para garantir que atualizações na base de conhecimento não degradem a qualidade?

---
*MBA em RAG & CAG Aplicados a Direito e Segurança Pública · Aula 8*